https://learn.microsoft.com/en-gb/azure/ai-foundry/agents/how-to/use-your-own-resources?view=foundry&preserve-view=true

In [1]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint="https://swedencdbeumz.services.ai.azure.com/api/projects/swedenprojectcdbeumz",
)

agent = project.agents.create_agent(
        model="gpt-4.1-mini",
        name="agentthread",
        instructions="You are a very helpful assistant"
)

agentid = agent.id

# create a thread
thread = project.agents.threads.create()
threadid = thread.id
print(f"Created thread, thread ID: {thread.id}")

message = project.agents.messages.create(
    thread_id=thread.id,
    role="user",
    content="Tell me a joke about Microsoft",
)
run = project.agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)

#get the messages from run.thread_id
messages = project.agents.messages.list(thread_id=run.thread_id)
for msg in messages:
    print(f"{msg.role:} {msg.content[0]['text']['value']}")
    print(f"Run completed with status: {run.status}")
    

Created thread, thread ID: thread_ZHs5NteuLZXZvKPwvawW0JSg
MessageRole.AGENT Sure! Here's a light-hearted joke about Microsoft:

Why did Microsoft Office go to therapy?

Because it had too many issues to Excel with!
Run completed with status: RunStatus.COMPLETED
MessageRole.USER Tell me a joke about Microsoft
Run completed with status: RunStatus.COMPLETED


#### Threads

In [2]:
import json
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential

#threadid = "thread_NRYrh5r7AQrWV3Co3IOZYhcj"

cosmos_uri = "https://eumzcosmosdb.documents.azure.com:443/"

credential = DefaultAzureCredential()
cosmos_client = CosmosClient(cosmos_uri, credential)

DATABASE_NAME = "enterprise_memory"
CONTAINER_NAME = "1b4c823b-83fa-4306-b2dd-7c6462d17153-thread-message-store"

db= cosmos_client.get_database_client(DATABASE_NAME)
container = db.get_container_client(CONTAINER_NAME)

results = container.query_items(
        query='SELECT TOP 10 * FROM c WHERE c.thread_id = "' + threadid + '"',
        enable_cross_partition_query=True,
            populate_index_metrics=True,
            populate_query_metrics=True)

res = ""
for result in results: 
    for key in ["kind", "content"]:
        if key in result:
            res += json.dumps(result[key], indent=2) + "\r\n"

print(res)



"enterprise_user_store"
{
  "type": "text",
  "text": "Tell me a joke about Microsoft"
}
"enterprise_user_store"
"enterprise_user_store"
{
  "type": "text",
  "text": "Sure! Here's a light-hearted joke about Microsoft:\n\nWhy did Microsoft Office go to therapy?\n\nBecause it had too many issues to Excel with!"
}



#### Agents

In [3]:
import json
from azure.cosmos import CosmosClient
from azure.identity import DefaultAzureCredential

#id = "ent_msg_UlXvng8h3dOXwcYxs3sztClB"
id = agentid

cosmos_uri = "https://eumzcosmosdb.documents.azure.com:443/"

credential = DefaultAzureCredential()
cosmos_client = CosmosClient(cosmos_uri, credential)

DATABASE_NAME = "enterprise_memory"
CONTAINER_NAME = "1b4c823b-83fa-4306-b2dd-7c6462d17153-agent-entity-store"

db= cosmos_client.get_database_client(DATABASE_NAME)
container = db.get_container_client(CONTAINER_NAME)

results = container.query_items(
        query='SELECT TOP 10 * FROM c WHERE c.payload.id = "' + id + '"',
        enable_cross_partition_query=True,
            populate_index_metrics=True,
            populate_query_metrics=True)


for result in results: 
    ret = result
    break
ret


{'hard_deleted_at': None,
 'internal_metadata': {},
 'azure_resource_location': None,
 '_etag': '"0200bff2-0000-4700-0000-697b3e5e0000"',
 'internal_note': None,
 'created_at': '2026-01-29T11:02:53.737284+00:00',
 'updated_at': '2026-01-29T11:02:53.737291+00:00',
 'deleted_at': None,
 'ttl': -1,
 'DEFAULT_TTL_SECONDS': 2592000,
 'id': 'ent_msg_rtvax3bIYB1cKbiKsDxbXppg',
 'partition_key_value': '1b4c823b-83fa-4306-b2dd-7c6462d17153_2026_01',
 'payload': {'type': 'agent',
  'id': 'asst_mPOUsFzMQlRhcMDnuderHQuO',
  'name': 'agentthread',
  'description': None,
  'instructions': 'You are a very helpful assistant',
  'tools': [],
  'metadata': {}},
 'kind': 'enterprise_entity_message',
 '_rid': 'u25+ALpCWXMRAAAAAAAAAA==',
 '_self': 'dbs/u25+AA==/colls/u25+ALpCWXM=/docs/u25+ALpCWXMRAAAAAAAAAA==/',
 '_attachments': 'attachments/',
 '_ts': 1769684574}